In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
!pip install scikit-learn --upgrade

In [ ]:
!pip install scikit-learn>=0.18

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_recall_f1_support, mean_squared_error
import uuid
import copy

# Load AI4I Dataset
df = pd.read_csv("ai4i2020.csv")

# Select Sensor Features
sensor_features = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]",
                   "Torque [Nm]", "Tool wear [min]"]
X = df[sensor_features]

# Define Multi-Label Targets
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
y = df[failure_types].astype(int)

# Add a Failure Sum column for stratification
df['Failure_Sum'] = y.sum(axis=1)

# Drop rows where Failure_Sum class occurs only once
failure_sum_counts = df['Failure_Sum'].value_counts()
valid_sums = failure_sum_counts[failure_sum_counts >= 2].index
valid_idx = df['Failure_Sum'].isin(valid_sums)

df_filtered = df[valid_idx].copy()
X_filtered = df_filtered[sensor_features]
y_filtered = df_filtered[failure_types]

# Create Label Encoding for Combined Failure Types
df_filtered['Failure_Type'] = y_filtered.apply(lambda row: '_'.join(map(str, row.values)), axis=1)
label_encoder = LabelEncoder()
df_filtered['Failure_Type_Encoded'] = label_encoder.fit_transform(df_filtered['Failure_Type'])

# Split into Temp & Holdout Sets
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X_filtered, df_filtered['Failure_Type_Encoded'], test_size=0.1,
    stratify=df_filtered['Failure_Sum'], random_state=42
)

# Simulate 3 Clients by Splitting Temp Data
client_splits = np.array_split(pd.concat([X_temp, y_temp], axis=1), 3)
clients_data = [(split[sensor_features], split['Failure_Type_Encoded']) for split in client_splits]

# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, input_shape=input_shape, return_sequences=False),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Federated Learning Functions
def aggregate_models(global_weights, client_weights, client_sizes):
    total_size = sum(client_sizes)
    new_weights = [np.zeros_like(w) for w in global_weights]
    for client_w, size in zip(client_weights, client_sizes):
        for i in range(len(new_weights)):
            new_weights[i] += client_w[i] * size / total_size
    return new_weights

def evaluate_model(model, X, y, label_encoder):
    predictions = model.predict(X)
    y_pred = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    precision, recall, f1, _ = precision_recall_f1_support(y, y_pred, average='weighted', zero_division=0)
    return accuracy, mse, precision, recall, f1

# Federated Learning Simulation
num_rounds = 5
num_clients = 3
scaler = StandardScaler()

# Initialize Global Model
input_shape = (1, len(sensor_features))
num_classes = len(label_encoder.classes_)
global_model = create_lstm_model(input_shape, num_classes)
global_weights = global_model.get_weights()

# Training Loop
for round in range(num_rounds):
    print(f"\nRound {round + 1}/{num_rounds}")
    client_weights = []
    client_sizes = []

    for client_id in range(num_clients):
        print(f"Training Client {client_id + 1}")

        # Get Client Data
        X_client, y_client = clients_data[client_id]

        # Handle Class Imbalance
        oversampler = RandomOverSampler(sampling_strategy="not majority", random_state=42)
        X_resampled, y_resampled = oversampler.fit_resample(X_client, y_client)

        # Scale Data
        X_scaled = scaler.fit_transform(X_resampled)
        X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

        # Create and Train Local Model
        local_model = create_lstm_model(input_shape, num_classes)
        local_model.set_weights(global_weights)
        local_model.fit(X_lstm, y_resampled, epochs=5, batch_size=16, verbose=0)

        # Collect Weights and Size
        client_weights.append(local_model.get_weights())
        client_sizes.append(len(y_resampled))

    # Aggregate Weights
    global_weights = aggregate_models(global_weights, client_weights, client_sizes)
    global_model.set_weights(global_weights)

    # Evaluate Global Model
    X_holdout_scaled = scaler.transform(X_holdout)
    X_holdout_lstm = X_holdout_scaled.reshape((X_holdout_scaled.shape[0], 1, X_holdout_scaled.shape[1]))
    accuracy, mse, precision, recall, f1 = evaluate_model(global_model, X_holdout_lstm, y_holdout, label_encoder)

    print(f"Global Model Evaluation - Round {round + 1}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

# Save Final Model
global_model.save("federated_lstm_model.keras")

ImportError: cannot import name 'precision_recall_f1_support' from 'sklearn.metrics' (/usr/local/lib/python3.12/dist-packages/sklearn/metrics/__init__.py)

In [ ]:
#hellow world
